# TFA Validation of Pareto-Optimal Solutions for *Rhodopseudomonas palustris* PHB Synthesis

> Clean, minimal notebook to validate Pareto-optimal operating points using **Thermodynamics-based Flux Analysis (TFA)**, focusing on PHB synthesis.
>
> **What this notebook does**
> 1. Loads the *R. palustris* genome-scale model (COBRApy).
> 2. Applies medium/exchange constraints for each Pareto-optimal solution.
> 3. Runs **standard FBA** (for baseline) and **TFA** (pyTFA) with `PHBS_syn` as objective and growth fixed to 0.
> 4. Verifies that TFA is **really active** (ΔG variables + binary directionality).
> 5. Extracts fluxes for key PHB-pathway reactions (ACACT1r, ACACCT, AACOAR_syn, HACD1, HACD1i, HACD1_2, KAT1).
> 6. Saves a tidy CSV comparing FBA vs TFA for every Pareto point.

**Notes**
- You need a working installation of `cobra` and `pytfa (>=0.9.1)` and a MILP-capable solver (e.g., CPLEX/GUROBI/GLPK).
- Paths to your model (`SBML` or JSON), thermodynamic DB (`thermo_data.thermodb`), and Pareto CSV are configurable below.
- This notebook keeps only the code required for TFA validation and omits tutorials or unrelated snippets.


## 0) Environment & Dependencies
If needed, uncomment the installs (e.g., in Colab). You must have a MILP solver.

In [2]:
!pip3 install pytfa cobra catboost optlang modelseedpy


Requested pytfa from https://files.pythonhosted.org/packages/bb/ac/1d6a4a72f45bfa46a95d3cc9f29c113c5f4a4ece4ac7461b56ac61c2fa2c/pytfa-0.9.4-py2.py3-none-any.whl has invalid metadata: Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier
    python-version (>="3.6") ; extra == 'equilibrator'
                   ~^
Please use pip<24.1 if you need to use this version.
Requested pytfa from https://files.pythonhosted.org/packages/a3/b4/9d3a72b34043fc5e4b75afcefd2d386ebab18ca99690cfa1c4c01612c77d/pytfa-0.9.3-py2.py3-none-any.whl has invalid metadata: Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier
    python-version (>="3.6") ; extra == 'equilibrator'
                   ~^
Please use pip<24.1 if you need to use this version.
Requested pytfa from https://files.pythonhosted.org/packages/52/75/1b2114796a7f0b51ce973798805a5ccf5e929078bc65e59dc44b4f5601ea/pytfa-0.9.2-py2.py3-none-any.whl has invalid metadata: Expected matching RIGHT_PAR

In [3]:
# !pip install cobra pytfa optlang --quiet
# If you have GUROBI or CPLEX, also ensure the corresponding optlang interface is available.
import sys, os, json, pathlib
from typing import Dict, List
import pandas as pd
import numpy as np

import cobra
from cobra import Model
from cobra.io import read_sbml_model, load_json_model
from cobra.util.solver import set_objective

# pyTFA imports
from pytfa.thermo.tmodel import ThermoModel
from pytfa.io import load_thermoDB
from pytfa.utils.logger import get_bistream_logger
from pytfa.optim.relaxation import relax_dgo

import optlang

print('Python:', sys.version)
print('cobra version:', cobra.__version__)
try:
    import pytfa
    print('pytfa version:', pytfa.__version__)
except Exception as e:
    print('pyTFA not importable yet:', e)


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
cobra version: 0.30.0
pyTFA not importable yet: module 'pytfa' has no attribute '__version__'


In [4]:
import os
from pathlib import Path
from google.colab import drive

def mount_drive():
  drive.mount('/content/drive', force_remount=True)
  drive_folder = "metabolic_modelling/phb-optimization-rpalustris/"
  os.chdir('/content/drive/MyDrive/'+ drive_folder)
  global PROJECT_ROOT
  PROJECT_ROOT = Path(os.getcwd())

mount_drive()


Mounted at /content/drive


In [7]:
# Import path saved in src

import sys
sys.path.append(str(PROJECT_ROOT / "src"))
from src.paths import PHB_TFA_DIR, PHB_MODEL_TFA_DIR, DATA_DIR, MODELS_DIR, PHB_MODEL_DIR, PHB_CHECKPOINTS_DIR, PHB_RESULTS_DIR, PHB_FIGURES_DIR, PHB_GEM_EXPERIMENTAL_DIR, PHB_GEM_AUGMENTATION_DIR, PHB_CATBOOST_DIR, PHB_PARETO_DIR


## 1) Configuration — paths, IDs, and solver
Update the paths below to your files. Reaction IDs can be adjusted if they differ in your model.

In [11]:
# === PATHS (edit as needed) ===
MODEL_PATH = PHB_MODEL_DIR/ '02_model_rpalustris_PHB_constrained.xml'   # or .xml/.sbml/.json
THERMO_DB_PATH = PHB_MODEL_TFA_DIR /'thermo_data.thermodb'               # pyTFA thermodynamic DB file
PARETO_CSV = PHB_PARETO_DIR/'03_pareto_best_all_strategies_0.3978.csv'    # your Pareto points
OUT_CSV = PHB_TFA_DIR /'PARETO_FBA_vs_TFA_PHB.csv'
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

# === CORE REACTION IDs (edit if different in your model) ===
PHB_RXN_ID = 'PHBS_syn'       # PHB synthesis objective
BIOMASS_RXN_ID = 'BIOMASS__1' # Biomass reaction to be clamped to zero
KEY_RXNS = [
    'AACOAR_syn',    # Acetoacetyl-CoA reductase
    'ACACT1r',       # Thiolase condensation
    'HACD1',         # 3-hydroxyacyl-CoA dehydrogenase (acetoacetyl-CoA)
    'HACD1i',        # isoenzyme / variant
    'HACD1_2',       # second isoenzyme / variant
    'ACACCT',        # Acetyl-CoA:acetoacetyl-CoA transferase
    'KAT1'           # 3-ketoacyl-CoA thiolase (β-oxidation-related)
]

# === EXCHANGE MAPPING (adjust to your model's exchange IDs and CSV column names) ===
# Provide a mapping from human-friendly CSV columns to actual exchange reaction IDs in your model.
# Example columns you might have: 'acetate', 'lactate', 'butyrate', 'isobutyrate', 'malate', 'hexanoate', 'octanoate', 'nh4', 'hco3'
EX_MAP = {
    'acetate': 'EX_ac_e',
    'lactate': 'EX_lac__D_e',
    'butyrate': 'EX_but_e',
    'isobutyrate': 'EX_isobut_e',
    'malate': 'EX_mal__L_e',
    'hexanoate': 'EX_hxa_e',
    'octanoate': 'EX_oct_e',
    'nh4': 'EX_nh4_e',
    'hco3': 'EX_hco3_e'
}

# Columns in PARETO_CSV giving lower bounds (uptake; typically negative) for each component above.
# If your CSV has different names (e.g., 'EX_ac_e_lb'), edit here accordingly.
EX_COLS_LB = {k: f'{k}_con' for k in EX_MAP.keys()}
#EX_COLS_UB = {k: f'{k}_ub' for k in EX_MAP.keys()}  # optional; often 0 for uptake-only

# Default bounds if a column is missing
DEFAULT_LB = 0.0
DEFAULT_UB = 0.0

# === Thermodynamic / environmental settings ===
DEFAULT_PH = 7.0
DEFAULT_IONIC_STRENGTH = 0.25
DEFAULT_TEMPERATURE = 298.15  # K

# === Solver preference for MILP (TFA) ===
PREFERRED_SOLVERS = ['glpk']


## 2) Utility functions
- Solver selection
- Loading the model
- Applying Pareto solution constraints to exchanges
- FBA baseline
- Building and optimizing a ThermoModel (TFA)
- Verifying TFA activation
- Extraction of key fluxes

In [12]:
def choose_available_solver(preferred=PREFERRED_SOLVERS):
    for s in preferred:
        try:
            interface = optlang.interface_for_solver(s)
            return s
        except Exception:
            continue
    raise RuntimeError('No preferred solver found. Install gurobi/cplex/glpk (with MILP).')

def load_cobra_model(path: str) -> Model:
    path = pathlib.Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Model file not found: {path}')
    if path.suffix.lower() in {'.json'}:
        return load_json_model(str(path))
    else:
        return read_sbml_model(str(path))

def apply_exchange_bounds(model: Model, row: pd.Series, ex_map: Dict[str,str], ex_cols_lb: Dict[str,str], ex_cols_ub: Dict[str,str], default_lb=DEFAULT_LB, default_ub=DEFAULT_UB):
    for key, rxn_id in ex_map.items():
        lb_col = ex_cols_lb.get(key)
        ub_col = ex_cols_ub.get(key)
        lb = float(row.get(lb_col, default_lb)) if lb_col in row else default_lb
        ub = float(row.get(ub_col, default_ub)) if ub_col in row else default_ub
        if rxn_id in model.reactions:
            rxn = model.reactions.get_by_id(rxn_id)
            rxn.lower_bound = lb
            rxn.upper_bound = ub
        else:
            print(f'[WARN] Reaction {rxn_id} not in model — skipping')

def clamp_biomass_and_set_phb_objective(model: Model, biomass_id=BIOMASS_RXN_ID, phb_id=PHB_RXN_ID):
    if biomass_id in model.reactions:
        r = model.reactions.get_by_id(biomass_id)
        r.lower_bound = 0.0
        r.upper_bound = 0.0
    else:
        print(f'[WARN] Biomass reaction {biomass_id} not found')
    if phb_id in model.reactions:
        set_objective(model, model.reactions.get_by_id(phb_id))
    else:
        raise KeyError(f'PHB synthesis reaction {phb_id} not found')

def run_fba(model: Model):
    sol = model.optimize()
    return sol

def build_tmodel(cobra_model: Model, thermo_db_path: str, pH=DEFAULT_PH, ionic_strength=DEFAULT_IONIC_STRENGTH, temperature=DEFAULT_TEMPERATURE) -> ThermoModel:
    if not os.path.exists(thermo_db_path):
        raise FileNotFoundError(f'Thermo DB not found: {thermo_db_path}')
    tmodel = ThermoModel(cobra_model)
    thermo_data = load_thermoDB(thermo_db_path)
    tmodel.add_thermo(thermo_data, pH=pH, ionic_strength=ionic_strength, temperature=temperature)
    # Convert to add thermodynamic variables and constraints
    tmodel.convert()
    return tmodel

def verify_tfa_activation(tmodel: ThermoModel) -> Dict[str,int]:
    # Count ΔG variables and binary directionality variables
    n_dg = sum(1 for v in tmodel.variables if v.name.startswith('dG_'))
    n_bin = sum(1 for v in tmodel.variables if hasattr(v, 'type') and getattr(v, 'type', None) == 'binary')
    return {'n_dG_vars': n_dg, 'n_binary_vars': n_bin}

def run_tfa(tmodel: ThermoModel):
    sol = tmodel.optimize()
    # If infeasible due to tight ΔG bounds, try a relaxed ΔG optimization (optional)
    if sol.status != 'optimal':
        print('[INFO] TFA not optimal, attempting ΔG relaxation…')
        relax_dgo(tmodel)
        sol = tmodel.optimize()
    return sol

def extract_key_fluxes(model_or_tmodel, keys: List[str]) -> Dict[str, float]:
    fluxes = {}
    for r in keys:
        try:
            rxn = model_or_tmodel.reactions.get_by_id(r)
            fluxes[r] = float(model_or_tmodel.solution.fluxes[rxn.id])
        except Exception:
            fluxes[r] = np.nan
    return fluxes


## 3) Load model and Pareto table

In [14]:
print(optlang.available_solvers)

{'GUROBI': False, 'GLPK': True, 'MOSEK': False, 'CPLEX': False, 'COINOR_CBC': False, 'SCIPY': True, 'OSQP': True, 'HIGHS': True}


In [16]:
# Pick the best available solver for MILP (needed by TFA)
solver = "glpk" #choose_available_solver()
print('Selected solver:', solver)

# Load model
base_model = load_cobra_model(MODEL_PATH)
base_model.solver = solver

# Load Pareto CSV
pareto_df = pd.read_csv(PARETO_CSV)
print('Pareto points:', len(pareto_df))
pareto_df.head(3)

Selected solver: glpk
Pareto points: 5


,Unnamed: 0,Experiment,Biomass (mg dw/L),PHB (mg/L),% PHB,PHV (mg/L),% PHV,5-ALA (micromol/L),Q10 (mg/g dw),Carotenoids (mg/g dw),...,total_N_con_pfba,total_N_con_pfba_molar,C/N (molar base)_con_pfba,total_C_cost_con_pfba,total_hco3_cost_con_pfba,total_N_cost_con_pfba,total_cost_con_pfba,PHB_pred_flux,optimization_strategy,PHB_SYN_UB
0,164.0,21,1417,238.44,16.83,10.65,0.75,NaN,NaN,NaN,...,0.0,0.0,inf,0.005900,0.0,0.0,0.005900,0.398558,SUP,0.3978
1,97.0,21,1417,238.44,16.83,10.65,0.75,NaN,NaN,NaN,...,0.0,0.0,inf,0.002566,0.0,0.0,0.002566,0.397861,CON,0.3978
2,355.0,21,1417,238.44,16.83,10.65,0.75,NaN,NaN,NaN,...,0.0,0.0,inf,0.004905,0.0,0.0,0.004905,0.398093,SUP_and_CON,0.3978


## 4) Iterate Pareto points → FBA baseline and TFA validation
This section will:
1. Copy the base model
2. Apply exchange/medium bounds from each Pareto row
3. Clamp biomass to 0 and set PHB objective
4. Run FBA
5. Build ThermoModel and run TFA
6. Verify TFA is active (ΔG + binaries)
7. Collect and store results

In [19]:
EX_COLS_UB= {}

records = []

for idx, row in pareto_df.iterrows():
    mdl = base_model.copy()
    mdl.solver = solver
    # Apply Pareto medium/exchange constraints
    apply_exchange_bounds(mdl, row, EX_MAP, EX_COLS_LB, EX_COLS_UB)
    # Clamp growth and set PHB objective
    clamp_biomass_and_set_phb_objective(mdl)

    # --- FBA baseline ---
    fba_sol = run_fba(mdl)
    fba_status = fba_sol.status
    fba_obj = float(fba_sol.objective_value) if fba_status == 'optimal' else np.nan
    fba_fluxes = extract_key_fluxes(mdl, KEY_RXNS) if fba_status == 'optimal' else {k: np.nan for k in KEY_RXNS}

    # --- TFA ---
    # Build thermodynamic model from the *constrained* COBRA model
    tmdl = build_tmodel(mdl, THERMO_DB_PATH)
    tmdl.solver = solver
    # Ensure PHB is objective and biomass is clamped (again, on tmdl object)
    clamp_biomass_and_set_phb_objective(tmdl)
    tfa_info = verify_tfa_activation(tmdl)
    tfa_sol = run_tfa(tmdl)
    tfa_status = tfa_sol.status
    tfa_obj = float(tfa_sol.objective_value) if tfa_status == 'optimal' else np.nan
    tfa_fluxes = extract_key_fluxes(tmdl, KEY_RXNS) if tfa_status == 'optimal' else {k: np.nan for k in KEY_RXNS}

    rec = {
        'pareto_index': idx,
        'fba_status': fba_status,
        'fba_obj_PHBS_syn': fba_obj,
        'tfa_status': tfa_status,
        'tfa_obj_PHBS_syn': tfa_obj,
        'n_dG_vars': tfa_info['n_dG_vars'],
        'n_binary_vars': tfa_info['n_binary_vars'],
    }
    # add fluxes
    for k,v in fba_fluxes.items():
        rec[f'FBA__{k}'] = v
    for k,v in tfa_fluxes.items():
        rec[f'TFA__{k}'] = v

    records.append(rec)

res_df = pd.DataFrame.from_records(records)
res_df.to_csv(OUT_CSV, index=False)
print('Saved:', OUT_CSV)
res_df.head(10)

[WARN] Reaction EX_lac__D_e not in model — skipping
[WARN] Reaction EX_isobut_e not in model — skipping
[WARN] Reaction EX_oct_e not in model — skipping


TypeError: PHBS_syn: 3hbcoa__R_c + phbg_c --> PHB_c + coa_c is not a valid objective for \* Problem: Unknown *\

Maximize
 obj: + PHBS_syn - PHBS_syn_reverse_8e587

Subject To
 gam6p_c: + GF6PTA - GF6PTA_reverse_21fb1 + PGAMT - PGAMT_reverse_52c23
 + GAMptspp - GAMptspp_reverse_3e396 + AGDC - AGDC_reverse_2ab9d = 0
 cgly_c: - AMPTASECG + AMPTASECG_reverse_11d5d + GTHRDH_syn
 - GTHRDH_syn_reverse_d99c5 + GGCLUT2 - GGCLUT2_reverse_70203
 + CGLYabcpp - CGLYabcpp_reverse_8e5ba + GTMLT - GTMLT_reverse_b58ef = 0
 achms_c: + HSERTA - HSERTA_reverse_23c8f - AHSERL2_1
 + AHSERL2_1_reverse_bd815 - AHSERL2 + AHSERL2_reverse_2d820 = 0
 pcox_u: + PSIum - PSIum_reverse_43c5e + 4 CYOOum
 - 4 CYOOum_reverse_37909 - 2 CBFCum + 2 CBFCum_reverse_e7502 - 2 CBFCu
 + 2 CBFCu_reverse_05bf9 + 2 CYO1b2_syn - 2 CYO1b2_syn_reverse_5dfba = 0
 octe9ACP_c: + DESAT18a - DESAT18a_reverse_fd859 - G3PAT1819Z_1
 + G3PAT1819Z_1_reverse_480d5 = 0
 fdp_c: - FBA + FBA_reverse_84806 - FBP + FBP_reverse_bf2c9 + PFK
 - PFK_rev

## 5) Quick sanity checks & summary plots (optional)
These cells give a high-level comparison of PHB objective values and the key reaction fluxes in FBA vs TFA.

In [20]:
import matplotlib.pyplot as plt
plt.figure(figsize=(5,4))
ok = res_df.query("tfa_status == 'optimal' and fba_status == 'optimal'")
plt.scatter(ok['fba_obj_PHBS_syn'], ok['tfa_obj_PHBS_syn'], c='tab:blue', alpha=0.7)
plt.plot([ok['fba_obj_PHBS_syn'].min(), ok['fba_obj_PHBS_syn'].max()],
         [ok['fba_obj_PHBS_syn'].min(), ok['fba_obj_PHBS_syn'].max()], 'k--', lw=1)
plt.xlabel('Objetivo PHB (FBA) [mmol gDW$^{-1}$ h$^{-1}$]')
plt.ylabel('Objetivo PHB (TFA) [mmol gDW$^{-1}$ h$^{-1}$]')
plt.title('Comparación rápida de objetivos PHB — FBA vs TFA')
plt.tight_layout()
plt.show()


NameError: name 'res_df' is not defined

<Figure size 500x400 with 0 Axes>

## 6) How to adapt if your CSV schema differs
If your Pareto CSV uses different column names or encodes bounds as JSON per row, you can adapt `apply_exchange_bounds` by parsing those fields.

**Example**: columns like `EX_ac_e_lb` and `EX_ac_e_ub` — simply set the `EX_COLS_LB/UB` dictionaries to match.

If you have a single JSON column named `bounds_json` with a dictionary of `{rxn_id: [lb, ub]}`, you can load and apply it.

In [ ]:
# Example alternative bounds handler (if you store JSON per row).
# def apply_exchange_bounds_from_json(model: Model, row: pd.Series, column='bounds_json'):
#     if column not in row or pd.isna(row[column]):
#         return
#     d = json.loads(row[column])
#     for rxn_id, bounds in d.items():
#         try:
#             rxn = model.reactions.get_by_id(rxn_id)
#             lb, ub = bounds
#             rxn.lower_bound = float(lb)
#             rxn.upper_bound = float(ub)
#         except Exception:
#             print(f'[WARN] Could not set bounds for {rxn_id}')


---
### Provenance
This notebook follows your manuscript's description of TFA deployment for validating Pareto-optimal PHB strategies in *R. palustris*, including: assembling a pyTFA `ThermoModel`, enforcing ΔG-based constraints, clamping growth, optimizing `PHBS_syn`, and verifying ΔG and binary directionality variables are active.
